# Реальная лабораторная 07 — Wholesale и Abalone

Ненадзорная сегментация, масштабирование и робастный анализ измерений.

> **Как работать:** выполняй ячейки последовательно. Сначала сформулируй ожидание, затем запускай код и объясняй результат. Не удаляй строки и не исправляй значения автоматически до выяснения их смысла.

**Данные:** скачиваются из UCI Machine Learning Repository через `download_uci_datasets.py`. Первый запуск требует интернета; после сохранения CSV лабораторная выполняется локально.

## 0. Исследовательский контракт

Запиши единицу наблюдения, целевую аудиторию анализа, момент принятия решения и то, какие выводы допустимы. Разделяй описательное исследование, прогноз и причинные утверждения.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

SEED = 42
RNG = np.random.default_rng(SEED)
DATA_DIR = Path("data/raw")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 60)

def load_real(slug):
    path = DATA_DIR / f"{slug}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден {path}. Сначала выполните: "
            f"python download_uci_datasets.py --only {slug}"
        )
    return pd.read_csv(path, low_memory=False)

def section(title, text):
    display(Markdown(f"### {title}\n\n{text}"))

## Часть A. Wholesale Customers — ненадзорная сегментация

In [ ]:
wholesale = load_real("wholesale_customers")
print(wholesale.shape); display(wholesale.head()); display(wholesale.describe(include="all").T)
spend_cols=[c for c in wholesale.columns if c.lower() in {"fresh","milk","grocery","frozen","detergents_paper","delicassen"}]

### Асимметрия и масштаб

In [ ]:
if spend_cols:
    display(wholesale[spend_cols].skew().sort_values(ascending=False).to_frame("skew"))
    fig,axes=plt.subplots(2,3,figsize=(14,7))
    for c,ax in zip(spend_cols,axes.ravel()): sns.histplot(wholesale[c], bins=30, ax=ax, color="#627f96"); ax.set_title(c)
    plt.tight_layout(); plt.show()

### PCA как средство исследования, а не доказательство кластеров

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
if spend_cols:
    X=wholesale[spend_cols].apply(pd.to_numeric,errors="coerce").fillna(0)
    Xlog=np.log1p(X.clip(lower=0))
    Z=StandardScaler().fit_transform(Xlog)
    pca=PCA(n_components=2,random_state=SEED)
    emb=pca.fit_transform(Z)
    print("Explained variance ratio:", pca.explained_variance_ratio_)
    plt.figure(figsize=(7,5)); plt.scatter(emb[:,0],emb[:,1],alpha=.6); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Wholesale spend profile"); plt.show()
    km=KMeans(n_clusters=3,random_state=SEED,n_init=20).fit(Z)
    print("Cluster sizes:", pd.Series(km.labels_).value_counts().sort_index().to_dict())
    display(X.assign(cluster=km.labels_).groupby("cluster").median())

## Часть B. Abalone — измерения и регрессионный target

In [ ]:
abalone=load_real("abalone")
print(abalone.shape); display(abalone.head()); display(abalone.describe(include="all").T)
rings=next((c for c in abalone.columns if c.lower()=="rings"),None)
if rings:
    abalone[rings]=pd.to_numeric(abalone[rings],errors="coerce")
    display(abalone[rings].value_counts().sort_index().to_frame("n"))
weight_cols=[c for c in abalone.columns if "weight" in c.lower()]
if weight_cols:
    display(abalone[weight_cols].corr(method="spearman"))

## Самостоятельные задания

1. Сравни сегменты Wholesale по Channel и Region.
2. Сравни PCA на исходных значениях и log1p-преобразованных. Почему меняется геометрия?
3. Измени число кластеров и random_state; оцени устойчивость сегментации.
4. Для Abalone исследуй связь размеров, весов и Rings.
5. Почему Rings — не то же самое, что непосредственно измеренный возраст?
6. Подготовь отчёт, где отделены наблюдаемые группы от бизнес-интерпретации.

## Итоговый отчёт

В финальной версии notebook должны быть: **вопрос → единица анализа → происхождение и ограничения данных → проверка качества → описательная статистика → закономерности → неопределённость → риски смещения/утечки → выводы → следующие проверки**. Каждый вывод должен ссылаться на таблицу, график или расчёт.